# Statistics in Python — Standalone Practice Notebook

Nine compact sections in the style of your review notebook: **core playbook → worked examples → unanswered practice for Tom**. Uses the same Titanic dataset. Run setup first.

In [1]:
from scipy import stats
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.anova import anova_lm

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)


---

## Worked methods and independent practice

## 1 — Chi-square test of independence
**Core playbook**
- `pd.crosstab(index=..., columns=...)`: observed cell counts, **not** normalized percentages.
- `stats.chi2_contingency(table, correction=False)`: returns `statistic`, `pvalue`, `dof`, `expected_freq`. `correction=False` disables Yates correction for 2×2 tables.
- Cramér’s V: `sqrt(chi2 / (n * min(r-1, c-1)))`; descriptive association effect size.
- Check expected counts and independence of observations; small sparse tables may need an exact test.

In [2]:
tab = pd.crosstab(df["sex"], df["survived"])
result = stats.chi2_contingency(tab, correction=False)
n = tab.to_numpy().sum()
cramers_v = np.sqrt(result.statistic / (n * min(tab.shape[0] - 1, tab.shape[1] - 1)))
pd.Series({"chi2": result.statistic, "df": result.dof, "p": result.pvalue, "Cramer's V": cramers_v})


chi2          2.630506e+02
df            1.000000e+00
p             3.711748e-59
Cramer's V    5.433514e-01
dtype: float64

**Variation:** A 3×2 table uses exactly the same test; `dof=(rows-1)*(columns-1)`.

In [3]:
tab_class = pd.crosstab(df["pclass"], df["survived"])
stats.chi2_contingency(tab_class, correction=False)


Chi2ContingencyResult(statistic=np.float64(102.88898875696056), pvalue=np.float64(4.549251711298793e-23), dof=2, expected_freq=array([[133.09090909,  82.90909091],
       [113.37373737,  70.62626263],
       [302.53535354, 188.46464646]]))

**Practice for Tom (no answers):**
- [ ] Test whether embarkation port and survival are independent. Report χ², df, p, and Cramér’s V.
- [ ] Test class × sex. Which cells have the largest observed-minus-expected counts?
- [ ] Explain why a row-normalized crosstab is not the right input.

## 2 — One-sample and two-sample proportion z-tests
**Core playbook**
- `proportions_ztest(count, nobs, value=p0)`: test one sample against a hypothesized proportion.
- `proportions_ztest(count=[x1,x2], nobs=[n1,n2], value=0)`: test difference in two independent proportions.
- `proportion_confint(count, nobs, method="wilson")`: Wilson CI for an individual proportion.
- Count successes (`sum()` for 0/1 outcomes) and eligible nonmissing observations (`count()`).
- For very small counts, consider exact binomial methods instead of a z approximation.

In [4]:
s = df["survived"].dropna()
x, n = int(s.sum()), len(s)
z, p = proportions_ztest(count=x, nobs=n, value=0.40)
pd.Series({
    "survival rate": x / n,
    "z": z, "p": p,
    "Wilson CI": proportion_confint(x, n, method="wilson"),
})


survival rate                                     0.383838
z                                                -0.991978
p                                                 0.321208
Wilson CI        (0.3524694250340098, 0.41620468146882916)
dtype: object

In [5]:
groups = df.groupby("sex")["survived"].agg(["sum", "count"])
rates = groups["sum"] / groups["count"]
z, p = proportions_ztest(groups["sum"], groups["count"])
pd.Series({
    "z": z, "p": p,
    "female - male": rates["female"] - rates["male"],
})


z                1.621883e+01
p                3.711748e-59
female - male    5.531301e-01
dtype: float64

**Practice for Tom:**
- [ ] Test whether first-class survival differs from 50%. Include a Wilson interval.
- [ ] Compare second-class versus third-class survival proportions. Report direction and magnitude.
- [ ] Compare female and male survival within third class only.

## 3 — One-sample and two-sample t-tests + Cohen’s d
**Core playbook**
- `stats.ttest_1samp(x, popmean=...)`: one mean versus a reference.
- `stats.ttest_ind(a, b, equal_var=False)`: **Welch** independent-samples t-test (preferred default with unequal variances).
- `result.confidence_interval()`: CI for the mean or mean difference, respectively.
- One-sample `d=(mean(x)-mu0)/sd(x)`; independent-groups pooled-SD Cohen’s d uses pooled within-group SD (even if inference uses Welch).
- Independent observations; t-tests concern means, not proportions. Pairing requires `ttest_rel`, not `ttest_ind`.

In [6]:
age = df["age"].dropna()
result = stats.ttest_1samp(age, popmean=30)
pd.Series({
    "mean": age.mean(), "t": result.statistic, "p": result.pvalue,
    "95% CI": result.confidence_interval(), "Cohen's d": (age.mean() - 30) / age.std(ddof=1),
})


mean                                        29.699118
t                                           -0.553458
p                                            0.580123
95% CI       (28.631790041821507, 30.766445252296133)
Cohen's d                                   -0.020713
dtype: object

In [7]:
female = df.loc[df["sex"] == "female", "age"].dropna()
male = df.loc[df["sex"] == "male", "age"].dropna()
result = stats.ttest_ind(female, male, equal_var=False)
n1, n2 = len(female), len(male)
pooled_sd = np.sqrt(((n1 - 1) * female.var(ddof=1) + (n2 - 1) * male.var(ddof=1)) / (n1 + n2 - 2))
pd.Series({
    "female - male": female.mean() - male.mean(),
    "Welch t": result.statistic, "df": result.df, "p": result.pvalue,
    "95% CI": result.confidence_interval(),
    "Cohen's d": (female.mean() - male.mean()) / pooled_sd,
})


female - male                                  -2.810936
Welch t                                        -2.525898
df                                            560.050557
p                                               0.011815
95% CI           (-4.9967983473435, -0.6250732113583837)
Cohen's d                                      -0.194215
dtype: object

**Practice for Tom:**
- [ ] Test whether mean fare differs from 25; compute one-sample d.
- [ ] Compare mean fare of survivors and nonsurvivors using Welch; report d.
- [ ] Explain how missing age changes the sample and why fare skew matters.

## 4 — One-way ANOVA + effect sizes
**Core playbook**
- `smf.ols('y ~ C(group)', data=df).fit()`: categorical predictor in OLS.
- `anova_lm(model, typ=2)`: ANOVA table (`sum_sq`, `df`, `F`, `PR(>F)`).
- `eta² = SS_effect / SS_total`; `partial eta² = SS_effect / (SS_effect + SS_error)` (same for one-way ANOVA).
- ANOVA omnibus F does not identify which pairs differ. Inspect residuals, variance assumptions, and group sizes.

In [8]:
model = smf.ols("age ~ C(pclass)", data=df).fit()
aov = anova_lm(model, typ=2)
ss_group, ss_error = aov.loc["C(pclass)", "sum_sq"], aov.loc["Residual", "sum_sq"]
aov["eta2"] = np.nan
aov.loc["C(pclass)", "eta2"] = ss_group / (ss_group + ss_error)
aov


,sum_sq,df,F,PR(>F),eta2
C(pclass),20929.627754,2.0,57.443484,7.487984e-24,0.139107
Residual,129527.008190,711.0,NaN,NaN,NaN


**Variation:** For pairwise comparisons after a significant omnibus test, use Tukey HSD (`statsmodels.stats.multicomp.pairwise_tukeyhsd`) with suitable assumptions.

In [9]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

complete = df[["age", "pclass"]].dropna()
pairwise_tukeyhsd(endog=complete["age"], groups=complete["pclass"])


**Practice for Tom:**
- [ ] Run a one-way ANOVA for fare by embarkation port; report eta².
- [ ] Inspect group means and identify which pairwise comparisons differ.
- [ ] Explain what you would do about highly skewed fares.

## 5 — Factorial ANOVA: Type III SS + partial eta²
**Core playbook**
- Formula: `y ~ C(A, Sum) * C(B, Sum)` includes **both main effects and interaction**.
- `C(..., Sum)` requests sum-to-zero coding. Use it for interpretable Type III main-effect tests; treatment coding gives different Type III tests.
- `anova_lm(model, typ=3)` yields Type III SS.
- `partial_eta² = SS_term / (SS_term + SS_residual)` for each term.
- Type III main effects in interaction models are conditional on model specification/coding; inspect interaction and cell means first. Unbalanced/missing cells need extra care.

In [10]:
model = smf.ols("age ~ C(sex, Sum) * C(pclass, Sum)", data=df).fit()
aov3 = anova_lm(model, typ=3)
aov3["partial_eta2"] = aov3["sum_sq"] / (aov3["sum_sq"] + aov3.loc["Residual", "sum_sq"])
aov3.loc["Residual", "partial_eta2"] = np.nan
aov3


,sum_sq,df,F,PR(>F),partial_eta2
Intercept,571072.727040,1.0,3217.668800,1.535185e-265,0.819649
"C(sex, Sum)",3061.934356,1.0,17.252252,3.672808e-05,0.023788
"C(pclass, Sum)",21567.622275,2.0,60.760619,4.434398e-25,0.146496
"C(sex, Sum):C(pclass, Sum)",481.844296,2.0,1.357459,2.579827e-01,0.003820
Residual,125656.031067,708.0,NaN,NaN,NaN


**Variation:** Three-way factorial models use `A * B * C`; that expansion includes all lower-order interactions. `+` includes only additive terms.

**Practice for Tom:**
- [ ] Fit fare ~ sex × passenger class using Type III SS and sum contrasts.
- [ ] Report F, p, and partial eta² for both main effects and the interaction.
- [ ] Explain what changes when you replace `*` with `+`.
- [ ] Explain why unequal cell sizes matter for interpretation.

## 6 — Correlation: Pearson and Spearman
**Core playbook**
- `stats.pearsonr(x,y)`: linear correlation, r, p, and `.confidence_interval()`.
- `stats.spearmanr(x,y)`: rank-based monotonic association, rho and p.
- `df[[x,y]].dropna()` ensures **paired** complete cases.
- Check scatterplots/outliers; correlation does not imply causation.

In [11]:
pair = df[["age", "fare"]].dropna()
pearson = stats.pearsonr(pair["age"], pair["fare"])
spearman = stats.spearmanr(pair["age"], pair["fare"])
pd.DataFrame({
    "statistic": [pearson.statistic, spearman.statistic],
    "p": [pearson.pvalue, spearman.pvalue],
    "95% CI": [pearson.confidence_interval(), None],
}, index=["Pearson r", "Spearman rho"])


,statistic,p,95% CI
Pearson r,0.096067,0.010216,"(0.022855492985710376, 0.16825303648200288)"
Spearman rho,0.135051,0.000296,None


**Variation:** `df[["age","fare","sibsp"]].corr(method="pearson")` produces a correlation matrix (no p-values).

**Practice for Tom:**
- [ ] Correlate fare and family size (`sibsp + parch + 1`) with both methods.
- [ ] Compare Pearson versus Spearman age–fare correlations and explain any difference.
- [ ] Recalculate age–fare correlation separately within each passenger class.

## 7 — Multiple linear regression (OLS)
**Core playbook**
- `smf.ols('y ~ x1 + x2 + C(group)', data=df).fit()` fits adjusted linear regression.
- `model.summary()` gives coefficients, SEs, t-tests, R² and adjusted R².
- `model.params`, `.pvalues`, `.conf_int()`, `.predict(new_df)` extract results.
- `*` adds interactions; `:` adds **only** an interaction.
- Check linearity, influential points, residual variance and collinearity. Coefficients are conditional associations, not automatically causal.

In [12]:
model = smf.ols("fare ~ age + sibsp + parch + C(pclass) + C(sex)", data=df).fit()
report = pd.DataFrame({"estimate": model.params, "p": model.pvalues})
report[["ci_low", "ci_high"]] = model.conf_int()
report.round(3)


,estimate,p,ci_low,ci_high
Intercept,94.939,0.000,83.376,106.503
C(pclass)[T.2],-68.363,0.000,-76.969,-59.758
C(pclass)[T.3],-78.275,0.000,-86.165,-70.384
C(sex)[T.male],-5.648,0.088,-12.147,0.852
age,-0.262,0.028,-0.495,-0.028
sibsp,4.484,0.014,0.896,8.071
parch,9.943,0.000,6.068,13.818


**Variation:** Add an interaction: `fare ~ age * C(pclass) + C(sex)`. Compare predictions at fixed covariate values rather than interpreting a categorical coefficient without its reference category.

In [13]:
model_interaction = smf.ols("fare ~ age * C(pclass) + C(sex)", data=df).fit()
model_interaction.params


Intercept             136.116572
C(pclass)[T.2]       -104.187122
C(pclass)[T.3]       -112.131988
C(sex)[T.male]         -9.894414
age                    -1.118974
age:C(pclass)[T.2]      0.958460
age:C(pclass)[T.3]      0.971657
dtype: float64

**Practice for Tom:**
- [ ] Regress age on class, sex, and family size; interpret one adjusted coefficient.
- [ ] Fit an age × sex interaction and explain its coefficient.
- [ ] Generate predicted age for two hypothetical passengers differing only in sex.

## 8 — Multiple logistic regression (binomial GLM)
**Core playbook**
- `smf.glm('binary_y ~ x1 + C(group)', data=df, family=sm.families.Binomial()).fit()` fits logistic regression.
- `.params` are **log-odds** coefficients; `np.exp(.params)` are **odds ratios**.
- `np.exp(model.conf_int())` gives OR confidence intervals.
- `.predict(new_df)` returns predicted probabilities (default response scale).
- Check separation, sparse categories, sample size, calibration, and functional form for numeric predictors.

In [14]:
model = smf.glm("survived ~ age + fare + C(pclass) + C(sex)", data=df, family=sm.families.Binomial()).fit()
or_table = pd.DataFrame({"OR": np.exp(model.params), "p": model.pvalues})
or_table[["ci_low", "ci_high"]] = np.exp(model.conf_int())
or_table.round(3)


,OR,p,ci_low,ci_high
Intercept,41.368,0.000,16.644,102.815
C(pclass)[T.2],0.279,0.000,0.151,0.515
C(pclass)[T.3],0.079,0.000,0.041,0.150
C(sex)[T.male],0.081,0.000,0.054,0.121
age,0.964,0.000,0.949,0.979
fare,1.001,0.817,0.996,1.005


**Variation:** `survived ~ age * C(sex) + C(pclass)` tests whether the age slope differs by sex on the **log-odds** scale.

**Practice for Tom:**
- [ ] Fit survival on sex and class; report ORs and 95% CIs.
- [ ] Add age and fare; describe how the sex OR changes.
- [ ] Predict survival for two passengers differing only in class.
- [ ] Explain why an OR of 2 does not mean twice the probability.

## 9 — Model interpretation, effect-size selection, and reporting
**Core playbook**
- Proportions: report group rates, **percentage-point difference**, test and CI; Wilson CI describes a single proportion, not the difference.
- Means: report mean difference, t/df/p, CI, and Cohen’s d.
- ANOVA: F/df/p and eta² or partial eta².
- Chi-square: χ²/df/p and Cramér’s V.
- Correlation: r or rho, p, CI when available.
- OLS: adjusted coefficient, CI, p, and model R²; logistic GLM: OR, CI, and predicted probability.
- Effect sizes measure magnitude, not proof of practical importance or causality.

In [15]:
model = smf.ols("age ~ C(sex) + C(pclass)", data=df).fit()
report = pd.DataFrame({"estimate": model.params, "p": model.pvalues})
report[["ci_low", "ci_high"]] = model.conf_int()
report.round(3)


,estimate,p,ci_low,ci_high
Intercept,35.744,0.0,33.522,37.965
C(sex)[T.male],4.585,0.0,2.524,6.646
C(pclass)[T.2],-8.490,0.0,-11.255,-5.725
C(pclass)[T.3],-13.871,0.0,-16.265,-11.476


**Practice for Tom — mixed interview drills (no answers):**
- [ ] A stakeholder asks whether survival differs by sex: choose a method, effect size, and reporting format.
- [ ] Determine whether mean age differs by class after controlling for sex. Explain ANOVA versus regression formulations.
- [ ] Compare age and fare as predictors of survival, and explain why their raw coefficients cannot be compared directly.
- [ ] Write a three-sentence interpretation of a fitted logistic model: statistical evidence, practical magnitude, and one caveat.